In [1]:
import pandas as pd
import os
import numpy as np
import multiprocessing
cores = multiprocessing.cpu_count()
from IPython.display import clear_output

# Stopwords & punctuation
import string
from nltk.corpus import stopwords
punct = string.punctuation
stopw = stopwords.words('english') + [x for x in punct]
stopw += [x.translate
    (str.maketrans('', '', punct)) for x in stopwords.words('english')]

stopw +=  ["'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']

from nltk.tokenize import word_tokenize

# Vectorization - Doc2vec
from gensim.models.doc2vec import TaggedDocument, Doc2Vec
from gensim.test.test_doc2vec import ConcatenatedDoc2Vec

# Vectorisation - Word2Vec
from gensim.models.word2vec import Word2Vec

# Classification
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

def precision_macro(y_true, y_pred):
    return precision_score(y_true, y_pred, average='macro', zero_division=0)

def recall_macro(y_true, y_pred):
    return recall_score(y_true, y_pred, average='macro', zero_division=0)

def f1_macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

scoring = {
                'accuracy': 'accuracy',
                'precision_macro': make_scorer(precision_macro),
                'recall_macro': make_scorer(recall_macro),
                'f1_macro': make_scorer(f1_macro)
            }

# Naive Bayes (Gaussian NB)
from sklearn.naive_bayes import GaussianNB


# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn.svm import LinearSVC

# Régression Logistique, Perceptron
from sklearn.linear_model import LogisticRegression, Perceptron

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Validation croisée - 5-fold cross-validation
stratified_kfold = StratifiedKFold(shuffle=True, random_state=42)

In [2]:
def tokenize_remove_stopwords(text):
 for token in word_tokenize(text):
    if token in stopw: continue
    yield (token)

**Charger les données**

In [3]:
folder = '../1-data/training_datasets'
datasets = os.listdir(folder)
datasets

['train_dataset_10pc.xlsx',
 'train_dataset_20pc.xlsx',
 'train_dataset_30pc.xlsx',
 'train_dataset_40pc.xlsx',
 'train_dataset_50pc.xlsx',
 'train_dataset_60pc.xlsx',
 'train_dataset_70pc.xlsx',
 'train_dataset_80pc.xlsx',
 'train_dataset_90pc.xlsx']

**Entraîner les modèles et sortir les résultats**

*Word2Vec*

In [4]:
def vectorize(document_tokenized, model, n):
    words_vecs = [model.wv[word] for word in document_tokenized if word in model.wv]
    if len(words_vecs) == 0:
        return np.zeros(n)
        
    words_vecs = np.array(words_vecs)
    return words_vecs.mean(axis=0)

In [5]:
# Load the test dataset
df_test = pd.read_excel('../1-data/test_dataset_10.xlsx')
corpus_test = df_test['text_post'].astype('str')
y_test = df_test['category'].astype('str')

training_report = []
test_report = []

for dataset in datasets:
    report = []
    ratio = int(dataset[14:-7])
    df = pd.read_excel(os.path.join(folder, dataset))

    corpus = df['text_post'].astype('str')
    corpus = [list(tokenize_remove_stopwords(doc)) for doc in corpus]

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 750, 1000, 2500, 5000, 10000, 15000]

    # Algorithmes à tester
    algorithms = {
        'Gaussian Naive Bayes' : GaussianNB(),
        'kNN (k=5)' : KNeighborsClassifier(n_neighbors=5, n_jobs = cores),
        'kNN (k=10)' : KNeighborsClassifier(n_neighbors=10, n_jobs = cores),
        'kNN (k=15)' : KNeighborsClassifier(n_neighbors=15, n_jobs = cores),
        'Support Vector Machines (SVM)' : LinearSVC(dual="auto"),
        'Logistic Regression': LogisticRegression(n_jobs = cores),
        'Perceptron': Perceptron(),
        'Random Forest': RandomForestClassifier(n_jobs = cores),
    }

    for n_features in n_features_values:
        model_w2v = Word2Vec(
                    corpus,
                    vector_size= n_features,
                    workers = cores,
        )

        X = np.array([vectorize(doc, model_w2v, n_features) for doc in corpus])
        X_test = np.array([vectorize(doc, model_w2v, n_features) for doc in corpus_test])
        print(len(X[0]))  
        y = df['category'].astype('str')
            

        for algorithm_name, algorithm in algorithms.items():
            # Perform cross-validation
            scores = cross_validate(
                estimator=algorithm,
                X=X,
                y=y,
                cv=stratified_kfold,
                scoring=scoring,
                return_train_score=False
            )

            # Collect results
            results = {
                '% Incels': int(ratio),
                'Algorithme': algorithm_name,
                'Modèle de vectorisation': 'CBOW (Word2Vec)',
                'N traits discr.': n_features,
                'accuracy': scores['test_accuracy'].mean(),
                'precision': scores['test_precision_macro'].mean(),
                'recall': scores['test_recall_macro'].mean(),
                'f1-score': scores['test_f1_macro'].mean()
            }

            report.append(results)

            print(algorithm_name, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n',
                  f"Accuracy: {results['accuracy']:.4f}, "
                  f"Precision: {results['precision']:.4f}, "
                  f"Recall: {results['recall']:.4f}, "
                  f"F1-score: {results['f1-score']:.4f}")
            clear_output(wait=True)

            # Fit the algorithm on the entire training set and evaluate on the test set
            # algorithm.fit(X, y)  # This does not involve cross-validation; it's a final fitting
            # predictions_test = algorithm.predict(X_test)

            # test_results = {
            #     '% Incels': int(ratio),
            #     'Algorithme': algorithm_name,
            #     'Modèle de vectorisation': 'TF-IDF',
            #     'N traits discr.': n_features,
            #     'accuracy': accuracy_score(y_test, predictions_test),
            #     'precision': precision_score(y_test, predictions_test, average='macro', zero_division=0),
            #     'recall': recall_score(y_test, predictions_test, average='macro', zero_division=0),
            #     'f1-score': f1_score(y_test, predictions_test, average='macro', zero_division=0)
            # }

            # test_report.append(test_results)

    # Export cross-validation results
    report_df = pd.DataFrame(report)
    report_df['nb_posts_incels'] = (report_df['% Incels'].apply(lambda x: x / 100) * df.shape[0]).astype(int)
    report_df['nb_posts_neutral'] = (report_df['% Incels'].apply(lambda x: 1 - (x / 100)) * df.shape[0]).astype(int)
    report_df['nb_posts_total'] = df.shape[0]

    # Reorganize column order
    report_df = report_df[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme',
                           'Modèle de vectorisation', 'N traits discr.', 'accuracy', 'precision', 'recall', 'f1-score']]

    training_report.append(report_df)

# Combine all cross-validation results into a single DataFrame
final_report_df = pd.concat(training_report)

# Save the final cross-validation report sorted by f1-score
final_report_df.sort_values(by='f1-score', ascending=False).to_csv(
    '../3-results/results_training_cbow.csv', index=False
)

# Combine all test results into a single DataFrame
test_report_df = pd.DataFrame(test_report)

# Save the test results report sorted by f1-score
# test_report_df.sort_values(by='f1-score', ascending=False).to_csv(
#     '../3-results/results_test.csv', index=False
# )

*Doc2Vec*

In [ ]:
for dataset in datasets:
    report = []
    ratio = int(dataset[14:-7])
    df = pd.read_excel(os.path.join(folder, dataset))

    corpus = df['text_post'].astype('str')
    corpus = [list(tokenize_remove_stopwords(doc)) for doc in corpus]
    corpus = [
        TaggedDocument(words, ['{}'.format(idx)])
        for idx, words in enumerate(corpus)
    ]

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 750, 1000, 2500, 5000, 10000, 15000]

    # Algorithmes à tester
    algorithms = {
        'Multinomial Naive Bayes' : MultinomialNB(),
        'kNN (k=5)' : KNeighborsClassifier(n_neighbors=5, n_jobs = cores),
        'kNN (k=10)' : KNeighborsClassifier(n_neighbors=10, n_jobs = cores),
        'kNN (k=15)' : KNeighborsClassifier(n_neighbors=15, n_jobs = cores),
        'Support Vector Machines (SVM)' : LinearSVC(dual="auto"),
        'Logistic Regression': LogisticRegression(n_jobs = cores),
        'Perceptron': Perceptron(),
        'Random Forest': RandomForestClassifier(n_jobs = cores),
    }
    for n_features in n_features_values:
        print(str(int(n_features/2)))
        model_dbow = Doc2Vec(
            corpus,
            dm = 0, # DBOW Algorithm
            vector_size= int(n_features/2),
            dm_concat = 0,
            dm_mean = 1,
            workers = cores,
        )

        model_dmm = Doc2Vec(
            corpus,
            dm = 1, # Distributed Memory Mean
            vector_size = int(n_features/2),
            dm_concat = 0,
            dm_mean = 1,
            workers = cores,
        )
        
        model = ConcatenatedDoc2Vec([model_dbow, model_dmm])

        X = [model.dv[str(doc.tags[0])] for doc in corpus]
        print(len(X[0]))  
        y = df['category'].astype('str')
            

        for algorithm_name, algorithm in algorithms.items():
            # Perform cross-validation
            scores = cross_validate(
                estimator=algorithm,
                X=X,
                y=y,
                cv=stratified_kfold,
                scoring=scoring,
                return_train_score=False
            )

            # Collect results
            results = {
                '% Incels': int(ratio),
                'Algorithme': algorithm_name,
                'Modèle de vectorisation': 'CBOW (Word2Vec)',
                'N traits discr.': n_features,
                'accuracy': scores['test_accuracy'].mean(),
                'precision': scores['test_precision_macro'].mean(),
                'recall': scores['test_recall_macro'].mean(),
                'f1-score': scores['test_f1_macro'].mean()
            }

            report.append(results)

            print(algorithm_name, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n',
                  f"Accuracy: {results['accuracy']:.4f}, "
                  f"Precision: {results['precision']:.4f}, "
                  f"Recall: {results['recall']:.4f}, "
                  f"F1-score: {results['f1-score']:.4f}")
            clear_output(wait=True)

            # Fit the algorithm on the entire training set and evaluate on the test set
            # algorithm.fit(X, y)  # This does not involve cross-validation; it's a final fitting
            # predictions_test = algorithm.predict(X_test)

            # test_results = {
            #     '% Incels': int(ratio),
            #     'Algorithme': algorithm_name,
            #     'Modèle de vectorisation': 'TF-IDF',
            #     'N traits discr.': n_features,
            #     'accuracy': accuracy_score(y_test, predictions_test),
            #     'precision': precision_score(y_test, predictions_test, average='macro', zero_division=0),
            #     'recall': recall_score(y_test, predictions_test, average='macro', zero_division=0),
            #     'f1-score': f1_score(y_test, predictions_test, average='macro', zero_division=0)
            # }

            # test_report.append(test_results)

    # Export cross-validation results
    report_df = pd.DataFrame(report)
    report_df['nb_posts_incels'] = (report_df['% Incels'].apply(lambda x: x / 100) * df.shape[0]).astype(int)
    report_df['nb_posts_neutral'] = (report_df['% Incels'].apply(lambda x: 1 - (x / 100)) * df.shape[0]).astype(int)
    report_df['nb_posts_total'] = df.shape[0]

    # Reorganize column order
    report_df = report_df[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme',
                           'Modèle de vectorisation', 'N traits discr.', 'accuracy', 'precision', 'recall', 'f1-score']]

    training_report.append(report_df)

# Combine all cross-validation results into a single DataFrame
final_report_df = pd.concat(training_report)

# Save the final cross-validation report sorted by f1-score
final_report_df.sort_values(by='f1-score', ascending=False).to_csv(
    '../3-results/results_training_dbow_dm.csv', index=False
)

# Combine all test results into a single DataFrame
test_report_df = pd.DataFrame(test_report)

# Save the test results report sorted by f1-score
# test_report_df.sort_values(by='f1-score', ascending=False).to_csv(
#     '../3-results/results_test.csv', index=False
# )

KeyboardInterrupt: 